# Phase 7 — Final Model Selection

## Objective

Select the final production-ready churn prediction model using the experiment results tracked in MLflow.

The project's primary metric is **Recall** because missing an actual churned customer is more costly than contacting an extra customer who may not churn.

This notebook does **not retrain** any model. It reuses:

- MLflow experiment results
- `src/models/tuned_xgboost.pkl`
- existing processed test data for a final artifact sanity check


In [1]:
from pathlib import Path
import os

import joblib
import mlflow
import pandas as pd


## 7.1 Configure Project Paths and MLflow

The tracking backend is the project-level SQLite database:

```text
mlflow.db
```

This avoids the deprecated file-store backend issue and keeps all experiment results in one clean location.


In [2]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

MLFLOW_DB_PATH = PROJECT_ROOT / "mlflow.db"
MLFLOW_TRACKING_URI = f"sqlite:///{MLFLOW_DB_PATH}"
EXPERIMENT_NAME = "Customer Churn Prediction"

os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_TRACKING_URI
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

print("Project root:", PROJECT_ROOT)
print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", EXPERIMENT_NAME)


Project root: /Users/anshumaansharma0404gmail.com/Desktop/ML-Strugles/customer-churn-prediction
MLflow tracking URI: sqlite:////Users/anshumaansharma0404gmail.com/Desktop/ML-Strugles/customer-churn-prediction/mlflow.db
Experiment: Customer Churn Prediction


## 7.2 Load MLflow Results

The notebook reads all runs from the `Customer Churn Prediction` experiment, keeps only runs with model evaluation metrics, and compares them using Recall, Precision, F1 Score, ROC-AUC, and Accuracy.


In [3]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if experiment is None:
    raise ValueError(f"MLflow experiment not found: {EXPERIMENT_NAME}")

runs_df = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

metric_columns = [
    "metrics.accuracy",
    "metrics.precision",
    "metrics.recall",
    "metrics.f1_score",
    "metrics.roc_auc",
]

comparison_df = (
    runs_df[["tags.mlflow.runName", "run_id", *metric_columns]]
    .rename(
        columns={
            "tags.mlflow.runName": "model_name",
            "metrics.accuracy": "accuracy",
            "metrics.precision": "precision",
            "metrics.recall": "recall",
            "metrics.f1_score": "f1_score",
            "metrics.roc_auc": "roc_auc",
        }
    )
    .dropna(subset=["recall"])
    .sort_values("recall", ascending=False)
    .reset_index(drop=True)
)

comparison_df


,model_name,run_id,accuracy,precision,recall,f1_score,roc_auc
0,Tuned XGBoost,4a5158e74bc944518edc3c5bd1969f6e,0.963475,0.853521,0.932308,0.891176,0.991809
1,Tuned XGBoost,b91b639ac5844fe9b54db03dcae6621d,0.963475,0.853521,0.932308,0.891176,0.991809
2,Tuned Random Forest,046c9d9f3c7d461bbd823836ad17f49c,0.943238,0.786885,0.886154,0.833575,0.980314
3,Tuned Random Forest,30aadb2aa1fa4ab084ba0b155a6d67de,0.943238,0.786885,0.886154,0.833575,0.980314
4,Baseline Random Forest,15c40feefbdb41058a1088b565472d1a,0.954590,0.862928,0.852308,0.857585,0.984810
5,Baseline XGBoost,c090b905451c4cac9892d4767e07fdde,0.965449,0.932203,0.846154,0.887097,0.991784
6,Baseline Logistic Regression,e7a6125588b24a6282c19be579253200,0.854886,0.530938,0.818462,0.644068,0.920677
7,Baseline Decision Tree,30efd014231548da8048f5c11878e08d,0.934847,0.818482,0.763077,0.789809,0.865372


## 7.3 Select Final Model

The final model is selected based on the highest Recall among valid tracked model runs.

Based on the completed tuning phase, **Tuned XGBoost** is expected to be the final model because it achieved the highest Recall while maintaining strong F1 Score and ROC-AUC.


In [4]:
best_run = comparison_df.iloc[0]

print("Selected final model:", best_run["model_name"])
print(f"Recall: {best_run['recall']:.4f}")
print(f"Precision: {best_run['precision']:.4f}")
print(f"F1 Score: {best_run['f1_score']:.4f}")
print(f"ROC-AUC: {best_run['roc_auc']:.4f}")

expected_final_model = "Tuned XGBoost"
if best_run["model_name"] != expected_final_model:
    raise ValueError(
        f"Expected {expected_final_model}, but MLflow comparison selected {best_run['model_name']}"
    )


Selected final model: Tuned XGBoost
Recall: 0.9323
Precision: 0.8535
F1 Score: 0.8912
ROC-AUC: 0.9918


## 7.4 Why Tuned XGBoost Was Selected

Tuned XGBoost is selected as the final production candidate because:

- It achieved the highest Recall among the tracked candidate models.
- It reduced missed churners compared with the baseline models.
- It maintained strong Precision, F1 Score, and ROC-AUC.
- It aligns with the business goal of identifying customers likely to churn before they leave.

In this project, Recall is more important than pure Accuracy because a false negative means the business fails to identify a customer who is actually at risk of churn.


## 7.5 Save Final Production Model

The already trained and tuned XGBoost model is loaded from:

```text
src/models/tuned_xgboost.pkl
```

It is saved as the final production model:

```text
src/models/final_model.pkl
```


In [5]:
tuned_xgboost_path = PROJECT_ROOT / "src" / "models" / "tuned_xgboost.pkl"
final_model_path = PROJECT_ROOT / "src" / "models" / "final_model.pkl"

if not tuned_xgboost_path.exists():
    raise FileNotFoundError(f"Missing tuned model: {tuned_xgboost_path}")

final_model = joblib.load(tuned_xgboost_path)
joblib.dump(final_model, final_model_path)

print("Final model saved to:", final_model_path)
print("Final model type:", type(final_model).__name__)


Final model saved to: /Users/anshumaansharma0404gmail.com/Desktop/ML-Strugles/customer-churn-prediction/src/models/final_model.pkl
Final model type: XGBClassifier


## 7.6 Sanity Check Final Model Artifact

This check confirms that the saved final model can be loaded and used for prediction on the existing processed test data.


In [6]:
X_test_path = PROJECT_ROOT / "data" / "processed" / "X_test_processed.csv"

X_test = pd.read_csv(X_test_path)
loaded_final_model = joblib.load(final_model_path)

sample_predictions = loaded_final_model.predict(X_test.head())
sample_probabilities = loaded_final_model.predict_proba(X_test.head())[:, 1]

pd.DataFrame(
    {
        "prediction": sample_predictions,
        "churn_probability": sample_probabilities,
    }
)


,prediction,churn_probability
0,0,0.004152
1,0,0.018629
2,0,0.144478
3,0,0.000569
4,0,0.002541


## Phase 7 Conclusion

**Tuned XGBoost** is confirmed as the final churn prediction model.

The production model artifact is now available at:

```text
src/models/final_model.pkl
```

This artifact should be used by the next phases:

- Phase 8: Model Explainability
- Phase 9: FastAPI Backend
